In [ ]:
import pandas as pd 

In [ ]:
data=pd.read_csv("housing.csv")

In [ ]:
data

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.info()


In [ ]:
data.describe()

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
data.hist(bins=50,figsize=(12,8))

In [ ]:
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
data

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
train_data, test_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

In [ ]:
train_data.head()

In [ ]:
test_data.head()

In [ ]:
train_data.shape

In [ ]:
test_data.shape

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()

housing_cat = cat_encoder.fit_transform(
    train_data[["ocean_proximity"]]
)

In [ ]:
housing_cat.toarray()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()

housing_cat = cat_encoder.fit_transform(
    train_data[["ocean_proximity"]]
)

housing_cat_test = cat_encoder.transform(
    test_data[["ocean_proximity"]]
)

In [ ]:

housing_cat_df = pd.DataFrame(
    housing_cat.toarray(),
    columns=cat_encoder.get_feature_names_out(["ocean_proximity"]),
    index=train_data.index
)

housing_cat_test_df = pd.DataFrame(
    housing_cat_test.toarray(),
    columns=cat_encoder.get_feature_names_out(["ocean_proximity"]),
    index=test_data.index
)


In [ ]:
train_data = train_data.drop("ocean_proximity", axis=1)
test_data = test_data.drop("ocean_proximity", axis=1)

train_data = pd.concat([train_data, housing_cat_df], axis=1)
test_data = pd.concat([test_data, housing_cat_test_df], axis=1)

In [ ]:
train_data

In [ ]:
X_train = train_data.drop("median_house_value", axis=1)
y_train = train_data["median_house_value"]

X_test = test_data.drop("median_house_value", axis=1)
y_test = test_data["median_house_value"]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

num_cols = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income"
]

cat_cols = ["ocean_proximity"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [ ]:
X_train_prepared = imputer.fit_transform(X_train)
X_test_prepared = imputer.transform(X_test)

In [ ]:
# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_prepared, y_train)

# Decision Tree
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(X_train_prepared, y_train)

# Random Forest
forest_reg = RandomForestRegressor(random_state=42)
forest_reg.fit(X_train_prepared, y_train)

In [ ]:
lin_preds = lin_reg.predict(X_train_prepared)

tree_preds = tree_reg.predict(X_train_prepared)

forest_preds = forest_reg.predict(X_train_prepared)

In [ ]:
from sklearn.metrics import mean_squared_error

lin_rmse = mean_squared_error(
    y_train, lin_preds
) ** 0.5

tree_rmse = mean_squared_error(
    y_train, tree_preds
) ** 0.5

forest_rmse = mean_squared_error(
    y_train, forest_preds
) ** 0.5

In [ ]:
print("Linear Regression RMSE:", lin_rmse)
print("Decision Tree RMSE:", tree_rmse)
print("Random Forest RMSE:", forest_rmse)

In [ ]:
from sklearn.model_selection import cross_val_score

lin_rmses = -cross_val_score(
    lin_reg,
    X_train_prepared,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=10
)

tree_rmses = -cross_val_score(
    tree_reg,
    X_train_prepared,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=10
)

forest_rmses = -cross_val_score(
    forest_reg,
    X_train_prepared,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=10
)

In [ ]:
print("Linear Regression:", lin_rmses.mean())
print("Decision Tree:", tree_rmses.mean())
print("Random Forest:", forest_rmses.mean())

In [ ]:
final_predictions = forest_reg.predict(X_test_prepared)

In [ ]:
from sklearn.metrics import mean_squared_error

final_rmse = mean_squared_error(
    y_test,
    final_predictions
) ** 0.5

print("Final Test RMSE:", final_rmse)

In [ ]:
one_house = X_test.iloc[[0]]

In [ ]:
one_house_prepared = imputer.transform(one_house)

In [ ]:
prediction = forest_reg.predict(one_house_prepared)

In [ ]:
print("Predicted house price:", prediction[0])

In [84]:
actual_price = y_test.iloc[0]

print("Predicted:", prediction[0])
print("Actual:", actual_price)

Predicted: 329135.14
Actual: 275200.0


In [85]:
predictions = forest_reg.predict(X_test_prepared)

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions
})

comparison.head(10)

,Actual,Predicted
0,275200.0,329135.14
1,234300.0,222697.00
2,155500.0,191151.00
3,60200.0,99773.00
4,140200.0,192143.00
5,114000.0,184875.00
6,127700.0,206921.00
7,71000.0,78842.00
8,165300.0,225150.00
9,163600.0,174826.00
